## Set up for agentic code

In [1]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [34]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

#Accessing environment variables
api_key = os.getenv('OPENAI_API_KEY')
print(f"API Key: {api_key[:8]}")


API Key: sk-proj-


In [35]:
from openai import OpenAI
client = OpenAI()
MODEL = "gpt-4.1-nano"
messages=[{"role":"user", "content":"What is the exponential of fibonacci of 3?"}]
response = client.chat.completions.create(
    model=MODEL,
    messages=messages)
result = response.choices[0].message.content
print(result)  
       

The Fibonacci number at position 3 is 2 (since the sequence starts with 0, 1, 1, 2, 3, ...).

Now, the exponential of 2 is:

\[ e^2 \approx 7.3891 \]

**Answer:** \(\boxed{e^2 \approx 7.3891}\)


In [36]:
system_prompt="""You are a helpful assistant that can perform mathematical calculations.
Respond  with exactlt one of the following formats:
1. FUNCTION_CALL: function_name|input
2. FINAL_ANSWER: [number]

where the function_name can be one of the following:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time.
"""

current_query="""Calculate the sum of exponentials of word "INDIA"""

prompt = f"{system_prompt}\n\n Query: {current_query}"
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role":"system", "content": prompt}])

print(response.choices[0].message.content)


FUNCTION_CALL: strings_to_chars_to_int|INDIA


In [37]:
import math

def strings_to_chars_to_int(string):
    return [ord(char) for char in string]

def int_list_to_exponential_sum(int_list):
    int_list = eval(int_list)
    return sum(math.exp(x) for x in int_list)

def fibonacci_numbers(n):
    if n <= 0:
        return []
    fib_sequence = [0, 1]
    for _ in range(2, n):
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence[:n]
       

In [38]:
response = response.choices[0].message.content
print(f"Response from model: {response}")

Response from model: FUNCTION_CALL: strings_to_chars_to_int|INDIA


In [39]:
_, function_info = response.split(":", 1)
_, function_info

('FUNCTION_CALL', ' strings_to_chars_to_int|INDIA')

In [40]:
func_name, params = [x.strip() for x in function_info.split("|", 1)]

func_name, params

('strings_to_chars_to_int', 'INDIA')

In [57]:
def function_caller(func_name, params):
    """Simple function caller that maps function names to actual functions"""
    function_map = {
        "strings_to_chars_to_int": strings_to_chars_to_int,
        "int_list_to_exponential_sum": int_list_to_exponential_sum,
        "fibonacci_numbers": fibonacci_numbers
    }
    
    if func_name in function_map:
        return function_map[func_name](params)
    else:
        return f"Function {func_name} not found"

In [42]:
iteration_result = function_caller(func_name, params)
print(f"Result of function call: {iteration_result}")

Result of function call: [73, 78, 68, 73, 65]


In [46]:
#Get model response 

system_prompt = """You are a math agent solving problems in iterations. Respond with EXACTLY ONE of these formats:
1. FUNCTION_CALL: python_function_name|input
2. FINAL_ANSWER: [number]

where python_function_name is one of the followin:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time."""

current_query= """Calculate the sum of exponentials of word "INDIA"""
iteration_1 = f"In the first iteration you called {func_name} with {params} parameters, and the function returned {iteration_result}. What should I do next?"
prompt = f"{system_prompt}\n\nQuery: {current_query}\n\n{iteration_1}"
response1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role":"system", "content": prompt}])


print(response1.choices[0].message.content)

FUNCTION_CALL: int_list_to_exponential_sum|[73, 78, 68, 73, 65]


In [47]:
response_text = response1.choices[0].message.content
_, function_info = response_text.split(":", 1)
func_name, params = [x.strip() for x in function_info.split("|", 1)]
iteration_result = function_caller(func_name, params)
iteration_result

7.59982224609308e+33

In [48]:
system_prompt = """You are a math agent solving problems in iterations. Respond with EXACTLY ONE of these formats:
1. FUNCTION_CALL: python_function_name|input
2. FINAL_ANSWER: [number]

where python_function_name is one of the followin:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time."""

current_query= """Calculate the sum of exponentials of word "INDIA"""
iteration_1 = f"In the first iteration you called strings_to_chars_to_int with INDIA parameters, and the function returned {iteration_result}. What should I do next?"
iteration_2 = f"In the first iteration you called {func_name} with {params} parameters, and the function returned {iteration_result}. What should I do next?"
prompt = f"{system_prompt}\n\nQuery: {current_query}\n\n{iteration_1}\n\n{iteration_2}"
response1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role":"system", "content": prompt}])


print(response1.choices[0].message.content)

FINAL_ANSWER: 7.59982224609308e+33


In [61]:
MAX_ITERATIONS = 4
last_response = None
iteration = 0
iteration_response = []

system_prompt = """You are a math agent solving problems in iterations. Respond with EXACTLY ONE of these formats:
1. FUNCTION_CALL: python_function_name|input b   
2. FINAL_ANSWER: [number]

where python_function_name is one of the followin:
1. strings_to_chars_to_int(string) It takes a word as input, and returns the ASCII INT values of characters in the word as a list
2. int_list_to_exponential_sum(list) It takes a list of integers and returns the sum of exponentials of those integers
3. fibonacci_numbers(int) It takes an integer, like 6, and returns first 6 integers in a fibonacci series as a list.
DO NOT include multiple responses. Give ONE response at a time."""

query= """Calculate the sum of exponentials of word "INDIA"""


while iteration < MAX_ITERATIONS:
    print(f"\n--- Iteration: {iteration+1} ---")
    if last_response == None:
        current_query = query
    else:
        current_query = current_query + "\n\n" + " ".join(iteration_response)  
        current_query = current_query + "  What should I do next?"

    #Get model response
    prompt = f"{system_prompt}\n\nQuery: {current_query}"
    response = client.chat.completions.create(
        model= "gpt-4.1-nano",
        messages=[{"role":"system", "content": prompt}]
    )

    response_text = response.choices[0].message.content
    print(f"Model response: {response_text}")

    if response_text.startswith("FUNCTION_CALL:"):
       
        _, function_info = response_text.split(":", 1)
        func_name, params = [x.strip() for x in function_info.split("|", 1)]
        iteration_result = function_caller(func_name, params)

        
    # Check if it's the final answer
    elif response_text.startswith("FINAL_ANSWER:"):
        print("\n=== Agent Execution Complete ===")
        break
        

    print(f"  Result: {iteration_result}")
    last_response = iteration_result
    iteration_response.append(f"In the {iteration + 1} iteration you called {func_name} with {params} parameters, and the function returned {iteration_result}.")

    iteration += 1






--- Iteration: 1 ---
Model response: 2. FUNCTION_CALL: strings_to_chars_to_int|input "INDIA"
  Result: [105, 110, 112, 117, 116, 32, 34, 73, 78, 68, 73, 65, 34]

--- Iteration: 2 ---
Model response: FINAL_ANSWER: 2074187362

=== Agent Execution Complete ===
